# Plantilla de EDA

Esta notebook guía un EDA manual sobre datos de ECG/tabulares. Ajusta las rutas y parámetros según tu dataset.


In [ ]:
import os, json, math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 200)
sns.set(style='whitegrid')

INPUT = './'  # CSV o Parquet
FILE_TYPE = 'auto'  # 'auto'|'csv'|'parquet'
SEP = ','
ENCODING = 'utf-8'
LIMIT_ROWS = None  # e.g., 200000
GLOB = None  # e.g., '*.csv' si INPUT es carpeta
TARGET = None  # p.ej. 'etiqueta'
TIME_COL = None  # p.ej. 'timestamp'

def infer_file_type(path: str, explicit: str | None):
    if explicit and explicit in {'csv','parquet'}: return explicit
    ext = os.path.splitext(path)[1].lower()
    if ext == '.csv': return 'csv'
    if ext in {'.parquet', '.pq'}: return 'parquet'
    return 'csv'

def list_files(path: str, ftype: str):
    if os.path.isdir(path):
        out = []
        for r,_,fs in os.walk(path):
            for f in fs:
                lf = f.lower()
                if ftype=='csv' and lf.endswith('.csv'): out.append(os.path.join(r,f))
                if ftype=='parquet' and (lf.endswith('.parquet') or lf.endswith('.pq')): out.append(os.path.join(r,f))
        return sorted(out)
    return [path]

FTYPE = infer_file_type(INPUT, None if FILE_TYPE=='auto' else FILE_TYPE)
FILES = list_files(INPUT, FTYPE)
FILES[:5], len(FILES)


In [ ]:
dfs = []
remaining = LIMIT_ROWS
for fp in FILES:
    if FTYPE=='csv':
        nrows = None if remaining is None else max(0, remaining)
        if nrows == 0: break
        df = pd.read_csv(fp, sep=SEP, encoding=ENCODING, nrows=nrows, low_memory=False, engine='python')
        dfs.append(df)
        if remaining is not None: remaining -= len(df)
    else:
        dfs.append(pd.read_parquet(fp))

df = pd.concat(dfs, ignore_index=True)
if LIMIT_ROWS is not None and FTYPE!='csv':
    df = df.head(LIMIT_ROWS)
df.shape, df.memory_usage(deep=True).sum() / 1024**2


In [ ]:
df.head()


In [ ]:
overview = {
    'num_rows': int(df.shape[0]),
    'num_columns': int(df.shape[1]),
    'memory_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 3),
    'columns': list(df.columns),
}
overview


In [ ]:
types = pd.DataFrame([
    {
        'columna': c,
        'dtype': str(df[c].dtype),
        'cardinalidad': int(df[c].nunique(dropna=True))
    } for c in df.columns
])
types.sort_values('columna')


In [ ]:
na = pd.DataFrame([
    {
        'columna': c,
        'nulos': int(df[c].isna().sum()),
        'nulos_%': round(df[c].isna().mean()*100, 3)
    } for c in df.columns
]).sort_values('nulos_%', ascending=False)
na.head(20)


In [ ]:
num_df = df.select_dtypes(include=[np.number])
desc = num_df.describe(percentiles=[0.25,0.5,0.75]).T
desc[['mean','std','min','25%','50%','75%','max']].head()


In [ ]:
for col in num_df.columns[:10]:
    fig, ax = plt.subplots(1,2, figsize=(10,3))
    sns.histplot(num_df[col].dropna(), kde=True, ax=ax[0]); ax[0].set_title(f'Hist {col}')
    sns.boxplot(x=num_df[col].dropna(), ax=ax[1]); ax[1].set_title(f'Box {col}')
    plt.show()


In [ ]:
corr = num_df.corr(method='pearson')
plt.figure(figsize=(max(6, int(corr.shape[0]*0.6)), max(5, int(corr.shape[1]*0.6))))
sns.heatmap(corr, cmap='coolwarm', center=0.0)
plt.title('Correlación (pearson)')
plt.show()
